<a href="https://colab.research.google.com/github/EMADUDDINAsdaq/federated-learning-fairness-xray/blob/main/notebooks/03_adaiffl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federated Learning — Method 3: Ada-IFFL (Cong et al. 2023)
Emaduddin Asdaq Syed Mohammed | CSC8639 MSc Data Science and AI

---
**This notebook runs Ada-IFFL only, on the full frozen dataset.**

**Algorithm 2 — Cong et al. 2023 (Ada-IFFL — Adaptive Individual Fairness FL).**
Extends q-FedAvg by computing the fairness exponent `q_k` adaptively per
client per round, instead of using a fixed, manually-tuned `q`:

`v_k = 1 − 2‖w_t − w̄_k‖ / (‖w_t‖ + ‖w̄_k‖)`  [normalised Frobenius
divergence between global and local model weights, Eq. 3]

`q_k = β · |1 − e^(−v_k)|`  [Eq. 4]

A larger model change (more divergence from the global model) produces a
larger `q_k`, giving that client more fairness-weighted influence — in
principle removing the need to hand-tune `q` as q-FedAvg requires.
β = 1.0 (fixed sensitivity parameter, dissertation S3.2.3).

**Scope note.** This implements Ada-IFFL specifically (Algorithm 2). The
extended Ada-FFL (Algorithm 4) adds a proximal regularisation term μ
requiring its own dataset-specific tuning — not implemented here, as it
would reintroduce the manual-parameter problem Ada-IFFL is meant to solve
(dissertation S2.1).

**Outcome (dissertation S4, Table 2 discussion).** This method did not
converge — global AUC 0.454, below the validity gate (>0.55) — and was
excluded from the fairness comparison. Per the dissertation's conclusion,
this is attributed to the Frobenius distance collapsing toward zero under
the small weight deviations typical of ResNet-18 fine-tuning, disabling
the adaptive fairness mechanism (consistent with ~0.07% predicted-positive
rate, i.e. near-total collapse to predicting "no finding").

Citation: Cong, Y. et al. 2023. Ada-FFL: Adaptive computing fairness
federated learning. *CAAI Transactions on Intelligence Technology*, 9(3),
573–584. https://doi.org/10.1049/cit2.12232

## Section 1 — Environment Setup

Installs Flower, imports dependencies, mounts Drive, confirms GPU.
Identical setup to the FedAvg and q-FedAvg notebooks for direct
comparability.

In [ ]:
pip install flwr protobuf

In [ ]:
!pip install "flwr[simulation]" protobuf -q
print("✓ Libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 25.5 MB/s eta 0:00:00
✓ Libraries installed


In [ ]:
import flwr as fl
print(f"flwr : {fl.__version__}")
print("✓ Flower working")

flwr : 1.32.1
✓ Flower working


In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
import flwr as fl
from flwr.common import (NDArrays, Scalar, Parameters,
                          parameters_to_ndarrays, ndarrays_to_parameters,
                          FitIns, FitRes, EvaluateIns, EvaluateRes)
from flwr.server.strategy import Strategy
from flwr.server.client_proxy import ClientProxy
from typing import Dict, List, Optional, Tuple
warnings.filterwarnings('ignore')

print(f"flwr  : {fl.__version__}")
print(f"torch : {torch.__version__}")
print(f"numpy : {np.__version__}")
print(f"GPU   : {torch.cuda.is_available()}")

flwr  : 1.32.1
torch : 2.11.0+cu128
numpy : 2.0.2
GPU   : True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import flwr.simulation
print("Simulation module loaded")

Simulation module loaded


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=== Session Initialisation ===")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"CUDA   : {torch.version.cuda}")
    print("\n✓ GPU ready")
else:
    print("\n⚠ No GPU — Runtime → Change runtime type → A100")

=== Session Initialisation ===
Device : cuda
GPU    : NVIDIA L4
Memory : 23.7 GB
CUDA   : 12.8

✓ GPU ready


## Section 2 — Dataset Download and Image Indexing

Re-downloads NIH ChestX-ray14 for this session.

In [ ]:
import shutil

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/dissertation/kaggle.json',
            '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✓ Kaggle credentials loaded")

os.system('pip install -q kaggle')
os.system('kaggle datasets download -d nih-chest-xrays/data '
          '--path /content/nih_kaggle --unzip --quiet')

DATASET_PATH = '/content/nih_kaggle'
print(f"✓ Dataset path: {DATASET_PATH}")

✓ Kaggle credentials loaded
✓ Dataset path: /content/nih_kaggle


## Section 3 — Load Frozen Splits

Loads the same 15 frozen CSVs used by every method notebook, so Ada-IFFL
trains and is evaluated on identical data to the other four strategies.

In [ ]:
SPLIT_DIR = '/content/drive/MyDrive/dissertation/splits'
HOSPITAL_NAMES = ['Hospital_A', 'Hospital_B', 'Hospital_C',
                  'Hospital_D', 'Hospital_E']
NUM_CLIENTS = 5

train_clients = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_train.csv') for n in HOSPITAL_NAMES}
val_clients   = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_val.csv')   for n in HOSPITAL_NAMES}
test_clients  = {n: pd.read_csv(f'{SPLIT_DIR}/{n}_test.csv')  for n in HOSPITAL_NAMES}
#Hospital_A_test.csv
for name in HOSPITAL_NAMES:
    print(f"{name}: {len(train_clients[name]):,} train / "
          f"{len(val_clients[name]):,} val / {len(test_clients[name]):,} test")

sample_path = train_clients['Hospital_A']['image_path'].iloc[0]
assert os.path.exists(sample_path), f"Path not found: {sample_path} — check Kaggle download completed"
print("✓ Image paths resolve correctly in this session")

Hospital_A: 52,332 train / 6,168 val / 3,005 test
Hospital_B: 9,074 train / 1,073 val / 508 test
Hospital_C: 29,813 train / 3,412 val / 1,711 test
Hospital_D: 1,276 train / 145 val / 74 test
Hospital_E: 2,997 train / 348 val / 168 test
✓ Image paths resolve correctly in this session


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 4 — GPU Optimisation

DataLoader and cuDNN settings, identical across all method notebooks.

In [ ]:
torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False

BATCH_SIZE  = 512
NUM_WORKERS = 4
PREFETCH    = 2

print(f"✓ BATCH_SIZE  : {BATCH_SIZE}")
print(f"✓ NUM_WORKERS : {NUM_WORKERS}")
print(f"✓ PREFETCH    : {PREFETCH}")

✓ BATCH_SIZE  : 512
✓ NUM_WORKERS : 4
✓ PREFETCH    : 2


## Section 5 — Transforms, Dataset

Same preprocessing and `Dataset` wrapper as every other method notebook.

In [ ]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

print(f"✓ Image size    : {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"✓ Normalisation : ImageNet mean/std")

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return image, label

print("✓ ChestXrayDataset defined")

✓ Image size    : 224×224
✓ Normalisation : ImageNet mean/std
✓ ChestXrayDataset defined


## Section 6 — Model

Same ResNet-18 architecture as every method notebook.

In [ ]:
ROUNDS = 10

def build_model():
    model    = models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

test_model = build_model()
params     = sum(p.numel() for p in test_model.parameters())
print(f"✓ ResNet-18 — ImageNet pretrained")
print(f"✓ Parameters  : {params:,}")
print(f"✓ Rounds      : {ROUNDS}")
del test_model

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 236MB/s]


✓ ResNet-18 — ImageNet pretrained
✓ Parameters  : 11,177,025
✓ Rounds      : 10


## Section 7 — Evaluation Function

Identical to the other notebooks: computes AUC/FNR overall and per
subgroup, skipping subgroups with fewer than 10 samples or a single class.

In [ ]:
def evaluate_client(model, dataframe, device):
    model = model.to(device)
    model.eval()

    loader = DataLoader(
        ChestXrayDataset(dataframe, transform=val_transform),
        batch_size         = BATCH_SIZE,
        shuffle            = False,
        num_workers        = NUM_WORKERS,
        pin_memory         = True,
        persistent_workers = True
    )

    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            probs = torch.sigmoid(
                model(images.to(device, non_blocking=True))
            ).cpu().numpy()
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs > 0.5).astype(int)

    def auc_fnr_for_mask(y_true, y_prob, y_pred):
        if len(y_true) < 10 or y_true.sum() == 0:
            return float('nan'), float('nan')
        try:
            auc = float(roc_auc_score(y_true, y_prob))
        except Exception:
            auc = float('nan')
        fn  = int(((y_pred == 0) & (y_true == 1)).sum())
        tp  = int(((y_pred == 1) & (y_true == 1)).sum())
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        return round(auc, 4), round(fnr, 4)

    auc, fnr = auc_fnr_for_mask(labels, probs, preds)
    acc      = round(float((preds == labels).mean() * 100), 2)
    metrics  = {'auc': auc, 'fnr': fnr, 'accuracy': acc}

    for sex in ['M', 'F']:
        mask = dataframe['Patient Sex'].values == sex
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{sex}'] = a
            metrics[f'fnr_{sex}'] = f

    for grp in ['0-20', '20-40', '40-60', '60-80', '80+']:
        mask = dataframe['Age Group'].values == grp
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{grp}'] = a
            metrics[f'fnr_{grp}'] = f

    return metrics

print("✓ evaluate_client() defined")
print("  Metrics : AUC + FNR (overall, per sex, per age group)")

✓ evaluate_client() defined
  Metrics : AUC + FNR (overall, per sex, per age group)


## Section 8 — Flower Base Client

Base per-hospital client, shared across method notebooks.

In [ ]:
class HospitalClient(fl.client.NumPyClient):

    def __init__(self, name: str, dataframe, val_dataframe, device):
        self.name          = name
        self.dataframe     = dataframe
        self.val_dataframe = val_dataframe
        self.device        = device
        self.model         = build_model().to(device)

    def get_parameters(self, config) -> NDArrays:
        return [v.cpu().numpy() for v in self.model.state_dict().values()]

    def set_parameters(self, parameters: NDArrays):
        state_dict = dict(zip(
            self.model.state_dict().keys(),
            [torch.tensor(p) for p in parameters]
        ))
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters: NDArrays, config: Dict) -> Tuple[NDArrays, int, Dict]:
        self.set_parameters(parameters)
        self.model.train()

        epochs = int(config.get('epochs', 3))
        lr     = float(config.get('lr', 1e-4))

        loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        criterion = nn.BCEWithLogitsLoss()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)

        total_loss, total_samples = 0.0, 0
        for _ in range(epochs):
            for images, labels in loader:
                images  = images.to(self.device, non_blocking=True)
                labels  = labels.to(self.device, non_blocking=True).unsqueeze(1)
                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples
        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss': float(avg_loss), 'client_name': self.name}
        )

    def evaluate(self, parameters: NDArrays, config: Dict) -> Tuple[float, int, Dict]:
        self.set_parameters(parameters)
        m = evaluate_client(self.model, self.val_dataframe, self.device)
        m['client_name'] = self.name
        return float(1.0 - (m['auc'] if not np.isnan(m['auc']) else 0.5)), \
               len(self.val_dataframe), m

print("✓ HospitalClient defined")
print("  fit()      → trains on train_clients")
print("  evaluate() → validates on val_clients after each round")

✓ HospitalClient defined
  fit()      → trains on train_clients
  evaluate() → validates on val_clients after each round


## Section 9 — Ada-IFFL (Cong et al. 2023)

Reuses the same pre-training-loss client as q-FedAvg (`F_k(w_t)` is
computed before local training, same as Li et al.'s Algorithm 2), since
Ada-IFFL only changes *how the exponent is chosen*, not the pre-loss
mechanism itself. The class name below is retained as
`QFedAvgHospitalClient` for that reason — this is the shared client, not
an Ada-IFFL-specific implementation.

In [ ]:
# Shared pre-training-loss client (identical to the q-FedAvg notebook).
# Kept under the same class name because the fit() logic — computing
# F_k(w_t) before local training — is unchanged between q-FedAvg and
# Ada-IFFL; only the server-side aggregation (AdaIFFLStrategy, below)
# differs, by computing q_k adaptively instead of using a fixed q.
class QFedAvgHospitalClient(HospitalClient):

    def fit(self, parameters, config):
        self.set_parameters(parameters)

        epochs    = int(config.get('epochs', 3))
        lr        = float(config.get('lr', 1e-4))
        criterion = nn.BCEWithLogitsLoss()

        # Pre-loss loader — num_workers=0, no persistent_workers,
        # avoids a Ray worker deadlock seen when two DataLoaders with
        # persistent workers run back-to-back in the same fit() call
        pre_loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size  = BATCH_SIZE,
            shuffle     = False,
            num_workers = 0,
            pin_memory  = False,
        )

        # F_k(w_t) — pre-training loss, read by AdaIFFLStrategy.aggregate_fit below
        self.model.eval()
        loss_sum, n = 0.0, 0
        with torch.no_grad():
            for images, labels in pre_loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True).unsqueeze(1)
                loss_sum += criterion(self.model(images), labels).item() * len(labels)
                n        += len(labels)
        loss_before = loss_sum / n

        train_loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        self.model.train()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)
        total_loss, total_samples = 0.0, 0

        for _ in range(epochs):
            for images, labels in train_loader:
                images  = images.to(self.device, non_blocking=True)
                labels  = labels.to(self.device, non_blocking=True).unsqueeze(1)
                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples
        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss': float(avg_loss), 'client_name': self.name,
             'loss_before': float(loss_before)}
        )

print("✓ QFedAvgHospitalClient defined")
print("  Computes F_k(w_t) before local training — shared with q-FedAvg")

✓ QFedAvgHospitalClient defined
  Computes F_k(w_t) before local training [Li et al. 2020 Algorithm 2]


In [ ]:
# CELL — q-FedAvg client factory

def make_qfedavg_client_fn(train_data_map, val_data_map, device):
    def client_fn(cid):
        name = HOSPITAL_NAMES[int(cid)]
        return QFedAvgHospitalClient(
            name          = name,
            dataframe     = train_data_map[name],
            val_dataframe = val_data_map[name],
            device        = device
        ).to_client()
    return client_fn

print("✓ Ada-IFFL client factory defined")

✓ Ada-IFFL client factory defined


In [ ]:
# beta (β) fixed at 1.0 — dissertation §3.2.3. Unlike q-FedAvg, q is not
# set here; it is computed per-client, per-round, inside _compute_q below.
BETA = 1.0

class AdaIFFLStrategy(Strategy):
    # Extends the q-FedAvg aggregation rule by replacing the fixed q with
    # an adaptive q_k, computed each round from the Frobenius distance
    # between each client's locally-trained weights and the current
    # global model (Cong et al. 2023, Eq. 3–4).

    def __init__(self, beta=1.0, L=1.0, num_clients=5):
        super().__init__()
        self.beta           = beta
        self.L              = L
        self.num_clients    = num_clients
        self._global_params = None

    def initialize_parameters(self, client_manager):
        ndarrays            = [v.cpu().numpy() for v in build_model().state_dict().values()]
        self._global_params = ndarrays
        return ndarrays_to_parameters(ndarrays)

    def configure_fit(self, server_round, parameters, client_manager):
        self._global_params = parameters_to_ndarrays(parameters)
        ins     = FitIns(parameters, {'epochs': 3, 'lr': 1e-4})
        sampled = client_manager.sample(num_clients=self.num_clients,
                                        min_num_clients=self.num_clients)
        return [(c, ins) for c in sampled]

    def _compute_q(self, w_t, w_bar_k):
        # Normalised Frobenius divergence score v_k (Eq. 3): 0 = identical
        # weights, 1 = maximally divergent. Larger divergence → larger q_k
        # (Eq. 4) → more fairness-weighted influence for that client.
        norm_diff = sum(np.linalg.norm(w - wb) for w, wb in zip(w_t, w_bar_k))
        norm_wt   = sum(np.linalg.norm(w)  for w  in w_t)
        norm_wb   = sum(np.linalg.norm(wb) for wb in w_bar_k)
        denom     = norm_wt + norm_wb
        v_k       = 1.0 - (2.0 * norm_diff / denom) if denom > 0 else 0.0
        v_k       = float(np.clip(v_k, 0.0, 1.0))
        q_k       = self.beta * abs(1.0 - np.exp(-v_k))
        return max(q_k, 1e-10)  # floored to avoid division errors in h_k below (dissertation §3.2.3)

    def aggregate_fit(self, server_round, results, failures):
        if not results:
            return None, {}

        w_t       = self._global_params
        sum_Delta = None
        sum_h     = 0.0

        for _, fit_res in results:
            w_bar_k = parameters_to_ndarrays(fit_res.parameters)
            loss_k  = max(float(fit_res.metrics.get('loss_before', 1.0)), 1e-10)

            q_k = self._compute_q(w_t, w_bar_k)  # per-client adaptive exponent, replaces q-FedAvg's fixed Q_PARAM

            delta_w_k = [self.L * (w - wb) for w, wb in zip(w_t, w_bar_k)]
            Delta_k   = [(loss_k ** q_k) * dw for dw in delta_w_k]
            norm_sq   = float(sum(np.sum(dw ** 2) for dw in delta_w_k))
            h_k       = (q_k * (loss_k ** (q_k - 1)) * norm_sq
                         + self.L * (loss_k ** q_k))

            sum_Delta = Delta_k if sum_Delta is None else \
                        [sd + dk for sd, dk in zip(sum_Delta, Delta_k)]
            sum_h    += h_k

        if sum_h == 0.0 or sum_Delta is None:
            return ndarrays_to_parameters(w_t), {}

        w_new               = [w - sd / sum_h for w, sd in zip(w_t, sum_Delta)]
        self._global_params = w_new
        return ndarrays_to_parameters(w_new), {}

    def aggregate_evaluate(self, server_round, results, failures):
        if not results:
            return None, {}

        print(f"\n── Round {server_round}/{ROUNDS} Validation ──")
        for _, res in results:
            name = res.metrics.get('client_name', '?')
            auc  = res.metrics.get('auc', float('nan'))
            fnr  = res.metrics.get('fnr', float('nan'))
            print(f"  {name:<14} AUC: {auc:.4f}  FNR: {fnr:.4f}")
        aucs       = [r.metrics.get('auc', float('nan')) for _, r in results]
        valid_aucs = [a for a in aucs if not (a != a)]
        if valid_aucs:
            print(f"  Mean AUC : {sum(valid_aucs)/len(valid_aucs):.4f} | "
                  f"Variance : {float(np.var(valid_aucs)):.6f}")

        total_loss = sum(r.loss * r.num_examples for _, r in results)
        total_n    = sum(r.num_examples for _, r in results)
        return total_loss / total_n, {}

    def configure_evaluate(self, server_round, parameters, client_manager):
        ins     = EvaluateIns(parameters, {})
        sampled = client_manager.sample(num_clients=self.num_clients,
                                        min_num_clients=self.num_clients)
        return [(c, ins) for c in sampled]

    def evaluate(self, server_round, parameters):
        return None  # centralised evaluation not used — all evaluation is client-side via aggregate_evaluate above

print(f"✓ AdaIFFLStrategy defined [Cong et al. 2023 Algorithm 2]")
print(f"  beta={BETA}  L=1.0")
print(f"  Individual adaptive q per client per round via Frobenius distance")
print(f"  Per-round validation output enabled")

✓ AdaIFFLStrategy defined [Cong et al. 2023 Algorithm 2]
  beta=1.0  L=1.0
  Individual adaptive q per client per round via Frobenius distance
  Per-round validation output enabled


### Run Training

Runs the 10-round Ada-IFFL simulation.

In [ ]:
# CELL — Run Ada-FFL simulation

import os, logging
os.environ['RAY_SILENT_MODE'] = '1'
logging.getLogger('flwr').setLevel(logging.ERROR)

print("=" * 50)
print("Ada-IFFL — Cong et al. 2023 Algorithm 2")
print(f"Rounds: {ROUNDS} | Epochs/round: 3 | Clients: {NUM_CLIENTS}")
print(f"beta={BETA}  L=1.0 | Split: 85/10/5 | Batch: {BATCH_SIZE}")
print("=" * 50)

t0              = time.time()
adaiffl_strategy = AdaIFFLStrategy(beta=BETA, L=1.0, num_clients=NUM_CLIENTS)

adaiffl_history = fl.simulation.start_simulation(
    client_fn        = make_qfedavg_client_fn(train_clients, val_clients, device),
    num_clients      = NUM_CLIENTS,
    config           = fl.server.ServerConfig(num_rounds=ROUNDS),
     strategy         = adaiffl_strategy,
    client_resources = {'num_gpus': 1.0}
)

print(f"\n{'='*50}")
print(f"✓ Ada-FFL complete in {(time.time()-t0)/60:.1f} minutes")
print(f"\nLoss per round:")
for rnd, loss in adaiffl_history.losses_distributed:
    print(f"  Round {rnd:>2} : {loss:.4f}")

Ada-IFFL — Cong et al. 2023 Algorithm 2
Rounds: 10 | Epochs/round: 3 | Clients: 5
beta=1.0  L=1.0 | Split: 85/10/5 | Batch: 512


2026-07-19 10:55:01,225	INFO worker.py:2012 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
(ClientAppActor pid=21340) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=21340) 
(ClientAppActor pid=21340)             This is a deprecated feature. It will be removed
(ClientAppActor pid=21340)             entirely in future versions of Flower.
(ClientAppActor pid=21340)         
(ClientAppActor pid=21340) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: Thi


── Round 1/10 Validation ──
  Hospital_A     AUC: 0.4627  FNR: 0.9995
  Hospital_B     AUC: 0.4150  FNR: 0.9981
  Hospital_E     AUC: 0.5225  FNR: 1.0000
  Hospital_C     AUC: 0.4634  FNR: 1.0000
  Hospital_D     AUC: 0.5244  FNR: 1.0000
  Mean AUC : 0.4776 | Variance : 0.001710


(ClientAppActor pid=21340) /usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
(ClientAppActor pid=21340)   warnings.warn(
(ClientAppActor pid=21340) /usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
(ClientAppActor pid=21340)   warnings.warn(
(ClientAppActor pid=21340) /usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
(ClientAppActor pid=21340)   warnings.warn(
(ClientAppActor pid=21340) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app impo


── Round 2/10 Validation ──
  Hospital_B     AUC: 0.4153  FNR: 0.9981
  Hospital_C     AUC: 0.4638  FNR: 1.0000
  Hospital_D     AUC: 0.5244  FNR: 1.0000
  Hospital_E     AUC: 0.5229  FNR: 1.0000
  Hospital_A     AUC: 0.4631  FNR: 0.9995
  Mean AUC : 0.4779 | Variance : 0.001705


...


── Round 8/10 Validation ──
  Hospital_A     AUC: 0.4648  FNR: 0.9992
  Hospital_E     AUC: 0.5236  FNR: 1.0000
  Hospital_D     AUC: 0.5253  FNR: 1.0000
  Hospital_B     AUC: 0.4162  FNR: 0.9981
  Hospital_C     AUC: 0.4657  FNR: 1.0000
  Mean AUC : 0.4791 | Variance : 0.001691


(ClientAppActor pid=21340) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=21340) 
(ClientAppActor pid=21340)             This is a deprecated feature. It will be removed
(ClientAppActor pid=21340)             entirely in future versions of Flower.
(ClientAppActor pid=21340)         
(ClientAppActor pid=21340) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=21340) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=21340)   self.pid = os.fork()
(ClientAppActor pid=21340) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app impor


── Round 9/10 Validation ──
  Hospital_C     AUC: 0.4659  FNR: 1.0000
  Hospital_A     AUC: 0.4651  FNR: 0.9992
  Hospital_E     AUC: 0.5237  FNR: 1.0000
  Hospital_B     AUC: 0.4165  FNR: 0.9981
  Hospital_D     AUC: 0.5256  FNR: 1.0000
  Mean AUC : 0.4794 | Variance : 0.001688


(ClientAppActor pid=21340) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=21340) 
(ClientAppActor pid=21340)             This is a deprecated feature. It will be removed
(ClientAppActor pid=21340)             entirely in future versions of Flower.
(ClientAppActor pid=21340)         
(ClientAppActor pid=21340) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=21340) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=21340)   self.pid = os.fork()
(ClientAppActor pid=21340) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app impor


── Round 10/10 Validation ──
  Hospital_A     AUC: 0.4653  FNR: 0.9992
  Hospital_E     AUC: 0.5239  FNR: 1.0000
  Hospital_C     AUC: 0.4663  FNR: 1.0000
  Hospital_B     AUC: 0.4165  FNR: 0.9981
  Hospital_D     AUC: 0.5251  FNR: 1.0000
  Mean AUC : 0.4794 | Variance : 0.001679

✓ Ada-FFL complete in 667.2 minutes

Loss per round:
  Round  1 : 0.5390
  Round  2 : 0.5386
  Round  3 : 0.5384
  Round  4 : 0.5381
  Round  5 : 0.5379
  Round  6 : 0.5376
  Round  7 : 0.5372
  Round  8 : 0.5370
  Round  9 : 0.5367
  Round 10 : 0.5365


### Final Test Evaluation and Save

Evaluates the final aggregated model on each hospital's held-out test set
and saves metrics and weights to Drive.

In [ ]:
# CELL — Evaluate Ada-IFFL per client

import gc, ray
if ray.is_initialized():
    ray.shutdown()
torch.cuda.empty_cache()
gc.collect()

adaiffl_metrics      = {}
adaiffl_final_params = adaiffl_strategy._global_params

for name, data in test_clients.items():
    model = build_model().to(device)
    model.load_state_dict(dict(zip(
        model.state_dict().keys(),
        [torch.tensor(p) for p in adaiffl_final_params]
    )))
    adaiffl_metrics[name] = evaluate_client(model, data, device)
    del model
    torch.cuda.empty_cache()

rows = []
for name in HOSPITAL_NAMES:
    m = adaiffl_metrics[name]
    rows.append({
        'Client'   : name,
        'AUC'      : m['auc'],
        'FNR'      : m['fnr'],
        'Accuracy' : m['accuracy'],
        'AUC_M'    : m.get('auc_M', 'N/A'),
        'FNR_M'    : m.get('fnr_M', 'N/A'),
        'AUC_F'    : m.get('auc_F', 'N/A'),
        'FNR_F'    : m.get('fnr_F', 'N/A'),
    })

df_adaiffl = pd.DataFrame(rows).set_index('Client')
print("=== Ada-IFFL Results — Cong et al. 2023 Algorithm 2 ===\n")
print(df_adaiffl.to_string())

valid_aucs = [adaiffl_metrics[n]['auc'] for n in HOSPITAL_NAMES
              if not (adaiffl_metrics[n]['auc'] != adaiffl_metrics[n]['auc'])]
auc_var = np.var(valid_aucs) if valid_aucs else float('nan')
print(f"\nAUC Variance (valid hospitals only) : {auc_var:.6f}")
print(f"Valid hospitals : {len(valid_aucs)}/5")

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=9501) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=9501) is multi-threaded, use of fork() may lead to deadlocks in the child.
 

=== Ada-IFFL Results — Cong et al. 2023 Algorithm 2 ===

               AUC     FNR  Accuracy   AUC_M   FNR_M   AUC_F   FNR_F
Client                                                              
Hospital_A  0.4474  1.0000     40.87  0.4419  1.0000  0.4547  1.0000
Hospital_B     NaN  0.9961      0.39     NaN  0.9963     NaN  0.9957
Hospital_C  0.4746  1.0000     92.11  0.4600  1.0000  0.5055  1.0000
Hospital_D  0.4362  1.0000     29.73  0.3818  1.0000  0.5132  1.0000
Hospital_E  0.4094  1.0000     88.69  0.3489  1.0000  0.4915  1.0000

AUC Variance (valid hospitals only) : 0.000547
Valid hospitals : 4/5


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# CELL — Save Ada-IFFL results to Drive

SAVE_DIR = '/content/drive/MyDrive/dissertation/results'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f'{SAVE_DIR}/adaiffl_full_metrics.json', 'w') as f:
    json.dump(adaiffl_metrics, f, indent=2, default=str)

torch.save(
    dict(zip(build_model().state_dict().keys(),
             [torch.tensor(v) for v in adaiffl_final_params])),
    f'{SAVE_DIR}/adaiffl_full_model.pth'
)

print("✓ Ada-IFFL metrics saved → adaiffl_full_metrics.json")
print("✓ Ada-IFFL model saved  → adaiffl_full_model.pth")

✓ Ada-IFFL metrics saved → adaiffl_full_metrics.json
✓ Ada-IFFL model saved  → adaiffl_full_model.pth
